# Example 10.1.2 / 1: Continuous variables, integral result

Companion notebook for **10.1.2 Total Unimodularity and Integral Flows** in the AMPL lab Documentation, section "Continuous variables, integral result". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Continuous variables, integral result

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set WORKERS;
set JOBS;
param cost {WORKERS, JOBS} >= 0;

var Assign {WORKERS, JOBS} >= 0;

minimize TotalCost:
    sum {w in WORKERS, j in JOBS} cost[w, j] * Assign[w, j];

subject to OneJobPerWorker {w in WORKERS}:
    sum {j in JOBS} Assign[w, j] = 1;

subject to OneWorkerPerJob {j in JOBS}:
    sum {w in WORKERS} Assign[w, j] = 1;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["WORKERS"] = ["Ana", "Bo", "Cy"]
ampl.set["JOBS"] = ["cut", "pack", "ship"]
ampl.param["cost"] = {("Ana", "cut"): 2, ("Ana", "pack"): 5, ("Ana", "ship"): 4, ("Bo", "cut"): 3, ("Bo", "pack"): 1, ("Bo", "ship"): 3, ("Cy", "cut"): 4, ("Cy", "pack"): 2, ("Cy", "ship"): 1}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result)
print(ampl.var["Assign"].to_dict())

HiGHS 1.14.0: optimal solution; objective 4
4 simplex iterations
0 barrier iterations
solved
{('Ana', 'cut'): 1, ('Ana', 'pack'): 0, ('Ana', 'ship'): 0, ('Bo', 'cut'): 0, ('Bo', 'pack'): 1, ('Bo', 'ship'): 0, ('Cy', 'cut'): 0, ('Cy', 'pack'): 0, ('Cy', 'ship'): 1}


### Pure AMPL: data.dat and model.run

The same in pure AMPL:

In [6]:
%%writefile data.dat
set WORKERS := Ana Bo Cy;
set JOBS := cut pack ship;
param cost: cut pack ship := Ana 2 5 4 Bo 3 1 3 Cy 4 2 1;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Assign;

HiGHS 1.14.0: optimal solution; objective 4
4 simplex iterations
0 barrier iterations
Assign :=
Ana cut    1
Ana pack   0
Ana ship   0
Bo  cut    0
Bo  pack   1
Bo  ship   0
Cy  cut    0
Cy  pack   0
Cy  ship   1
;

